# Steam Review Sentiment Analysis — Stacking Ensemble (NLP Project)

**Pipeline:** VADER Labeling → Preprocessing → TF-IDF + BERT Embeddings → Stacking (SVM + RF + BERT) → Meta-Learner

**3 Kelas:** Positive, Negative, Neutral

In [3]:
# Kode ini untuk install library yang diperlukan (uncomment jika belum)
!pip install pandas numpy matplotlib seaborn nltk scikit-learn wordcloud tqdm flask joblib sentence-transformers

In [ ]:
# Kode ini untuk import semua library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, os, warnings, joblib
warnings.filterwarnings('ignore')
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from wordcloud import WordCloud
from collections import Counter
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
print('Semua library berhasil di-import!')

In [5]:
# Kode ini untuk download NLTK resources
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('NLTK resources ready!')

---
## TAHAP 1 — Load Dataset

In [6]:
# Kode ini untuk memuat dataset bersih
df = pd.read_csv('steam_review_game_bersih.csv')
print(f'Dataset: {len(df):,} baris x {len(df.columns)} kolom')
print(f'Kolom: {list(df.columns)}')
df.head()

---
## TAHAP 2 — Exploratory Data Analysis

In [7]:
# Kode ini untuk menampilkan statistik dasar dan distribusi review
df['word_count'] = df['review'].astype(str).apply(lambda x: len(x.split()))

print(f'Total review: {len(df):,}')
print(f'Game unik: {df["appid"].nunique():,}')
print(f'Rata-rata kata/review: {df["word_count"].mean():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].hist(df['word_count'], bins=60, color='#66C0F4', edgecolor='#1B2838', alpha=0.85)
axes[0].set_title('Distribusi Jumlah Kata per Review', fontsize=15, fontweight='bold')
axes[0].set_xlim(0, df['word_count'].quantile(0.95))
axes[0].set_xlabel('Jumlah Kata'); axes[0].set_ylabel('Frekuensi')

top_games = df['name'].value_counts().head(10)
axes[1].barh(range(len(top_games)), top_games.values, color=plt.cm.viridis(np.linspace(0.3,0.9,10)))
axes[1].set_yticks(range(len(top_games))); axes[1].set_yticklabels(top_games.index)
axes[1].set_title('Top 10 Game (Jumlah Review)', fontsize=15, fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout(); plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight'); plt.show()

---
## TAHAP 3 — Pelabelan Sentimen (VADER) — 3 Kelas
Positive (compound ≥ 0.05), Negative (compound ≤ -0.05), Neutral (sisanya)

In [8]:
# Kode ini untuk melabeli sentimen menggunakan VADER (3 kelas: Positive, Negative, Neutral)
sia = SentimentIntensityAnalyzer()
print('Menjalankan VADER pada seluruh dataset...')
tqdm.pandas(desc='VADER')
df['vader_compound'] = df['review'].astype(str).progress_apply(lambda x: sia.polarity_scores(x)['compound'])
df['sentiment'] = df['vader_compound'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral'))

print(f'\nDistribusi Sentimen (3 Kelas):')
print(df['sentiment'].value_counts())
print(f'\nPersentase:')
print((df['sentiment'].value_counts(normalize=True) * 100).round(2))

In [9]:
# Kode ini untuk visualisasi distribusi sentimen 3 kelas
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
colors_s = {'Positive':'#4CAF50', 'Negative':'#F44336', 'Neutral':'#FFC107'}
sc = df['sentiment'].value_counts()

axes[0].pie(sc.values, labels=sc.index, colors=[colors_s[s] for s in sc.index],
            autopct='%1.1f%%', startangle=90, explode=[0.04]*3,
            textprops={'fontsize':13, 'fontweight':'bold'})
axes[0].set_title('Distribusi Sentimen', fontsize=15, fontweight='bold')

bars = axes[1].bar(sc.index, sc.values, color=[colors_s[s] for s in sc.index], edgecolor='#1B2838')
axes[1].set_title('Jumlah per Sentimen', fontsize=15, fontweight='bold')
for b,v in zip(bars, sc.values):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1500, f'{v:,}', ha='center', fontweight='bold')

axes[2].hist(df['vader_compound'], bins=50, color='#66C0F4', edgecolor='#1B2838', alpha=0.85)
axes[2].axvline(0.05, color='#4CAF50', ls='--', lw=2, label='+0.05')
axes[2].axvline(-0.05, color='#F44336', ls='--', lw=2, label='-0.05')
axes[2].set_title('VADER Compound Score', fontsize=15, fontweight='bold'); axes[2].legend()
plt.tight_layout(); plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

---
## TAHAP 4 — Text Preprocessing
Lowercase → Hapus URL/HTML/Special chars → Tokenisasi → Stopwords → Lemmatization

In [10]:
# Kode ini untuk mendefinisikan fungsi preprocessing dan mengaplikasikannya
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('Memulai preprocessing (estimasi 10-20 menit)...')
tqdm.pandas(desc='Preprocessing')
df['review_clean'] = df['review'].progress_apply(preprocess_text)
df = df[df['review_clean'].str.strip() != ''].reset_index(drop=True)
print(f'Selesai! Dataset: {len(df):,} baris')
df[['review', 'review_clean', 'sentiment']].head()

---
## TAHAP 5 — Visualisasi Setelah Preprocessing

In [11]:
# Kode ini untuk membuat Word Cloud per sentimen (Positif, Negatif, Netral)
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
cmaps = {'Positive': 'Greens', 'Negative': 'Reds', 'Neutral': 'YlOrBr'}
titles = {'Positive': ('POSITIF', '#388E3C'), 'Negative': ('NEGATIF', '#D32F2F'), 'Neutral': ('NETRAL', '#F57F17')}

for idx, sent in enumerate(['Positive', 'Negative', 'Neutral']):
    freq = Counter()
    for text in df[df['sentiment']==sent]['review_clean']:
        freq.update(str(text).split())
    wc = WordCloud(width=800, height=400, background_color='white', colormap=cmaps[sent], max_words=120)
    wc.generate_from_frequencies(dict(freq.most_common(200)))
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud — {titles[sent][0]}', fontsize=15, fontweight='bold', color=titles[sent][1])
    axes[idx].axis('off')

plt.tight_layout(); plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight'); plt.show()

In [12]:
# Kode ini untuk menampilkan top 20 kata per sentimen
fig, axes = plt.subplots(1, 3, figsize=(24, 9))
colors_bar = ['#4CAF50', '#F44336', '#FFC107']

for idx, sent in enumerate(['Positive', 'Negative', 'Neutral']):
    words = Counter()
    for text in df[df['sentiment']==sent]['review_clean']:
        words.update(str(text).split())
    top = words.most_common(20)
    axes[idx].barh([w for w,_ in top][::-1], [c for _,c in top][::-1], color=colors_bar[idx], edgecolor='#1B2838')
    axes[idx].set_title(f'Top 20 Kata — {sent}', fontsize=14, fontweight='bold')

plt.tight_layout(); plt.savefig('top_words.png', dpi=150, bbox_inches='tight'); plt.show()

---
## TAHAP 6 — Feature Extraction
1. **TF-IDF** (30K fitur, unigram+bigram) — untuk SVM & Random Forest
2. **BERT Embeddings** (MiniLM-L6-v2, 384 dimensi) — untuk BERT Classifier

In [13]:
# Kode ini untuk melakukan TF-IDF Vectorization
print('TF-IDF Vectorization...')
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=5, max_df=0.95, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df['review_clean'])
print(f'TF-IDF shape: {X_tfidf.shape}')

In [14]:
# Kode ini untuk menghasilkan BERT embeddings menggunakan sentence-transformers
print('Menghasilkan BERT embeddings (estimasi 20-40 menit di CPU)...')
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
X_bert = bert_model.encode(df['review_clean'].tolist(), show_progress_bar=True, batch_size=256)
print(f'BERT embeddings shape: {X_bert.shape}')

In [ ]:
# Kode ini untuk encode label dan split train/test
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])  # 0=Negative, 1=Neutral, 2=Positive
print(f'Label classes: {list(le.classes_)}')
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Train/Test split (80:20)
X_tfidf_train, X_tfidf_test, X_bert_train, X_bert_test, y_train, y_test = train_test_split(
    X_tfidf, X_bert, y, test_size=0.2, random_state=42, stratify=y)

print(f'\nTrain: {len(y_train):,} | Test: {len(y_test):,}')

---
## TAHAP 7 — Training Base Models
1. **Linear SVM** — pada fitur TF-IDF
2. **Random Forest** — pada fitur TF-IDF
3. **BERT Classifier** (Logistic Regression pada BERT embeddings)

In [ ]:
# Kode ini untuk melatih 3 base model
print('Training base models...\n')

# Base Model 1: Linear SVM
svm_model = LinearSVC(max_iter=2000, random_state=42, C=1.0)
svm_model.fit(X_tfidf_train, y_train)
svm_acc = accuracy_score(y_test, svm_model.predict(X_tfidf_test))
print(f'Linear SVM — Test Accuracy: {svm_acc:.4f}')

# Base Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_leaf=10,
                                   max_features='sqrt', n_jobs=-1, random_state=42)
rf_model.fit(X_tfidf_train, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_tfidf_test))
print(f'Random Forest — Test Accuracy: {rf_acc:.4f}')

# Base Model 3: BERT Classifier (LogReg on BERT embeddings)
bert_clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
bert_clf.fit(X_bert_train, y_train)
bert_acc = accuracy_score(y_test, bert_clf.predict(X_bert_test))
print(f'BERT Classifier — Test Accuracy: {bert_acc:.4f}')

---
## TAHAP 8 — Stacking Ensemble
Menggunakan **cross_val_predict** untuk mendapatkan out-of-fold predictions dari setiap base model,
lalu melatih **Meta-Learner** (Logistic Regression) pada prediksi yang di-stack.

```
          ┌──────────┐
Review →  │ SVM      │ → pred_svm  ─┐
          ├──────────┤               │    ┌──────────────┐
Review →  │ RF       │ → pred_rf   ─┼──→ │ Meta-Learner │ → Final Prediction
          ├──────────┤               │    └──────────────┘
Review →  │ BERT CLF │ → pred_bert ─┘
          └──────────┘
```

In [ ]:
# Kode ini untuk mendapatkan OOF predictions dan melatih meta-learner (STACKING)
print('Menghasilkan Out-of-Fold predictions (cv=3)...')
print('⏳ SVM OOF...')
svm_oof = cross_val_predict(LinearSVC(max_iter=2000, random_state=42, C=1.0),
                             X_tfidf_train, y_train, cv=3, method='decision_function')
print('⏳ Random Forest OOF...')
rf_oof = cross_val_predict(RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_leaf=10,
                            max_features='sqrt', n_jobs=-1, random_state=42),
                            X_tfidf_train, y_train, cv=3, method='predict_proba')
print('⏳ BERT Classifier OOF...')
bert_oof = cross_val_predict(LogisticRegression(max_iter=1000, random_state=42, C=1.0),
                              X_bert_train, y_train, cv=3, method='predict_proba')

# Stack OOF predictions sebagai fitur untuk meta-learner
X_stack_train = np.hstack([svm_oof, rf_oof, bert_oof])  # (n_train, 9)
print(f'\nStacked features shape: {X_stack_train.shape}')

# Latih Meta-Learner
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(X_stack_train, y_train)
print('Meta-Learner (Logistic Regression) berhasil dilatih!')

# Prediksi test set dengan stacking
svm_test_pred = svm_model.decision_function(X_tfidf_test)
rf_test_pred = rf_model.predict_proba(X_tfidf_test)
bert_test_pred = bert_clf.predict_proba(X_bert_test)
X_stack_test = np.hstack([svm_test_pred, rf_test_pred, bert_test_pred])

y_pred_stack = meta_learner.predict(X_stack_test)
y_proba_stack = meta_learner.predict_proba(X_stack_test)

stack_acc = accuracy_score(y_test, y_pred_stack)
stack_f1 = f1_score(y_test, y_pred_stack, average='weighted')
print(f'\n🏆 STACKING ENSEMBLE — Accuracy: {stack_acc:.4f} | F1: {stack_f1:.4f}')

---
## TAHAP 9 — Evaluasi Model

In [ ]:
# Kode ini untuk menampilkan classification report setiap model vs stacking
label_names = list(le.classes_)

all_results = {
    'Linear SVM': svm_model.predict(X_tfidf_test),
    'Random Forest': rf_model.predict(X_tfidf_test),
    'BERT Classifier': bert_clf.predict(X_bert_test),
    'STACKING ENSEMBLE': y_pred_stack
}

for name, preds in all_results.items():
    print(f'\n{"="*65}')
    print(f'  {name} — Accuracy: {accuracy_score(y_test, preds):.4f}')
    print(f'{"="*65}')
    print(classification_report(y_test, preds, target_names=label_names))

In [ ]:
# Kode ini untuk menampilkan confusion matrix semua model + stacking
fig, axes = plt.subplots(1, 4, figsize=(28, 6))
for idx, (name, preds) in enumerate(all_results.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[idx],
                xticklabels=label_names, yticklabels=label_names, annot_kws={'size':12})
    axes[idx].set_title(f'{name}\nAcc: {accuracy_score(y_test, preds):.4f}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Actual'); axes[idx].set_xlabel('Predicted')
plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Kode ini untuk perbandingan akurasi dan F1-score semua model
fig, ax = plt.subplots(figsize=(12, 7))
names = list(all_results.keys())
accs = [accuracy_score(y_test, all_results[n]) for n in names]
f1s = [f1_score(y_test, all_results[n], average='weighted') for n in names]
x = np.arange(len(names)); w = 0.3

b1 = ax.bar(x-w/2, accs, w, label='Accuracy', color='#66C0F4', edgecolor='#1B2838')
b2 = ax.bar(x+w/2, f1s, w, label='F1-Score', color='#4CAF50', edgecolor='#1B2838')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.12); ax.legend(fontsize=12); ax.grid(axis='y', alpha=0.3)
ax.set_title('Perbandingan Performa: Base Models vs Stacking', fontsize=16, fontweight='bold')
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.4f}',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

---
## TAHAP 10 — Simpan Semua Model

In [ ]:
# Kode ini untuk menyimpan semua model, vectorizer, dan meta-learner
os.makedirs('model', exist_ok=True)

joblib.dump(tfidf, 'model/tfidf_vectorizer.joblib')
joblib.dump(svm_model, 'model/svm_model.joblib')
joblib.dump(rf_model, 'model/rf_model.joblib')
joblib.dump(bert_clf, 'model/bert_classifier.joblib')
joblib.dump(meta_learner, 'model/meta_learner.joblib')
joblib.dump(le, 'model/label_encoder.joblib')

print(' Semua model berhasil disimpan di folder model/')
print('   - tfidf_vectorizer.joblib')
print('   - svm_model.joblib')
print('   - rf_model.joblib')
print('   - bert_classifier.joblib')
print('   - meta_learner.joblib')
print('   - label_encoder.joblib')
print(f'\n🏆 Stacking Ensemble Accuracy: {stack_acc:.4f}')

---
## TAHAP 11 — Testing Prediksi

In [ ]:
# Kode ini untuk menguji prediksi stacking pada review baru
def predict_stacking(text):
    cleaned = preprocess_text(text)
    feat_tfidf = tfidf.transform([cleaned])
    feat_bert = bert_model.encode([cleaned])
    
    s1 = svm_model.decision_function(feat_tfidf)
    s2 = rf_model.predict_proba(feat_tfidf)
    s3 = bert_clf.predict_proba(feat_bert)
    stacked = np.hstack([s1, s2, s3])
    
    pred = meta_learner.predict(stacked)[0]
    proba = meta_learner.predict_proba(stacked)[0]
    label = le.inverse_transform([pred])[0]
    return label, dict(zip(le.classes_, proba.round(4)))

tests = [
    "This game is absolutely amazing! Best game ever!",
    "Terrible, crashes every 5 minutes. Waste of money.",
    "It's okay, nothing special but playable.",
    "Over 2000 hours played, incredible community!",
    "DO NOT BUY. Devs abandoned it. Full of hackers.",
    "Decent graphics, average gameplay. Not bad not great.",
    "MASTERPIECE! The story made me cry. 10/10."
]

print('=== TEST PREDIKSI STACKING ENSEMBLE ===\n')
for t in tests:
    label, probs = predict_stacking(t)
    emoji = {'Positive':'😊','Negative':'😞','Neutral':'😐'}[label]
    print(f'Review  : "{t}"')
    print(f'Sentimen: {label} {emoji}')
    print(f'Proba   : {probs}\n')

print('Pipeline selesai! Jalankan app.py untuk Web App.')